In [ ]:
'''
Mini Project — Báo Cáo Tài Chính Cá Nhân
Đây là bài toán mở. Trước khi code, bạn cần thiết kế trước.
Yêu cầu chức năng:

Nhập thu nhập và các khoản chi tiêu trong tháng
Phân loại chi tiêu (ăn uống, đi lại, giải trí, tiết kiệm...)
Tính tỷ lệ từng khoản trên thu nhập
Lấy tỷ giá USD/VND từ API
Lưu báo cáo ra file JSON
In báo cáo đẹp ra console
'''
import requests
from requests.exceptions import Timeout, RequestException
import json

class BaoCaoChiTieu():
    '''
    Class bao cao chi tieu
    '''
    def __init__(self, thang, thu_nhap):
        self.thang = thang
        self.thu_nhap = thu_nhap
        self.chi_tieu = []
    def thong_ke_chi_tieu(self):
        '''
        Thong ke chi tieu
        '''
        while True:
            khoan_chi = {}
            print("\n1.Thêm 2.Xem danh sách  3.Thoát")
            lua_chon = input("Lựa chọn tính năng từ 1-3 theo danh sách trên: ")
            if lua_chon == "1":
                thoi_gian = input("Nhập thời gian chi tiêu: ")
                nhom = input("Nhập nhóm chi tiêu: ")
                mo_ta = input("Nhập mô tả chi tiêu: ")
                try:
                    so_tien = int(input("Nhập số tiền chi tiêu: "))
                except ValueError:
                    print("Nhập số tiền không hợp lệ. Vui lòng nhập lại số nguyên.")
                    continue
                khoan_chi["thoi_gian"] = thoi_gian
                khoan_chi["nhom"] = nhom
                khoan_chi["mo_ta"] = mo_ta
                khoan_chi["so_tien"] = so_tien
                self.chi_tieu.append(khoan_chi)
                print("Thêm thành công")
                continue
            elif lua_chon == "2":
                print("Danh sách chi tiêu hiện tại:")
                print(self.chi_tieu)
                continue
            elif lua_chon == "3":
                break
            else:
                print("Lựa chọn không hợp lệ. Vui lòng chọn lại.")
                continue

    def tong_tien_chi_tieu(self):
        '''
        Tong tien chi tieu
        '''
        tong_chi_nhom = {}
        for khoan in self.chi_tieu:
            nhom = khoan["nhom"]
            so_tien = khoan["so_tien"]
            if nhom not in tong_chi_nhom:
                tong_chi_nhom[nhom] = 0
            tong_chi_nhom[nhom] += so_tien
        return tong_chi_nhom

    def tinh_ty_le(self):
        '''
        Tinh ty le
        '''
        tong_nhom = self.tong_tien_chi_tieu()
        ty_le = {}
        if self.thu_nhap == 0:
            raise ValueError("Thu nhập phải lớn hơn 0")
        for nhom in tong_nhom:
            ty_le[nhom] = tong_nhom[nhom] / self.thu_nhap * 100
        return ty_le

    def lay_ty_gia(self):
        '''
        Lay ty gia
        '''
        url = "https://open.er-api.com/v6/latest/USD"
        try:
            response = requests.get(url, timeout = 5)
            response.raise_for_status()
            data = response.json()
            if data["result"] != "success":
                raise ValueError("Lấy dữ liệu không thành công")
            ty_gia = data["rates"]["VND"]
            return ty_gia
        except Timeout:
            print("Thời gian chờ đợi đã hết")
            return None
        except RequestException as e:
            print(f"Lỗi kết nối: {e}")
            return None

    def luu_bao_cao(self):
        '''
        Luu bao cao
        '''
        ten_file = f"bao_cao_thu_chi_thang_{self.thang}.json"
        du_lieu = {
            "thang": self.thang,
            "thu_nhap": self.thu_nhap,
            "chi_tieu": self.chi_tieu,
            "tong_chi": self.tong_tien_chi_tieu(),
            "ty_le_tung_nhom": self.tinh_ty_le(),
            "ty_gia": self.lay_ty_gia()}
        try:
            with open(ten_file, "w", encoding="utf-8") as file:
                json.dump(du_lieu, file,ensure_ascii = False, indent = 2)
            print(f"Lưu báo cáo thành công vào tệp {ten_file}")
        except Exception as e:
            print(f"Lỗi khi lưu báo cáo: {e}")

    def in_bao_cao(self):
        '''
        In bao cao
        '''
        tong_nhom = self.tong_tien_chi_tieu()
        ty_le = self.tinh_ty_le()
        ty_gia = self.lay_ty_gia()
        print("=" * 30)
        print(f"BÁO CÁO CHI TIÊU THÁNG {self.thang}")
        print("=" * 30)
        print(f"Thu nhập: {self.thu_nhap:,.0f}")
        print(f"Tỷ giá: {ty_gia:,.0f}")
        print("" * 30)
        print("Chi tiêu theo nhóm:")
        for nhom in tong_nhom:
            print(f"- {nhom}: {tong_nhom[nhom]:,.0f} VND ({ty_le[nhom]:.1f}%)")
        print("-"*30)
        print(f"Tổng chi: {sum(tong_nhom.values()):,.0f} VND")
        print(f"Còn dư: {self.thu_nhap - sum(tong_nhom.values()):,.0f} VND")
        print("="*30)


if __name__ == "__main__":
    thang = input("Nhập tháng (vd: 2026-04): ")
    thu_nhap = int(input("Nhập thu nhập: "))
    bao_cao = BaoCaoChiTieu(thang, thu_nhap)
    bao_cao.thong_ke_chi_tieu()
    bao_cao.in_bao_cao()
    bao_cao.luu_bao_cao()
